# LLM генерация cypher по вопросу и, если ошибка выполнения, -- исправление cypher силами LLM 
1. LLM generates Cypher using prompt_1 + schema + user question
2. Execute Cypher
3. If execution error, call LLM fixer and overwrite cypher

In [49]:
# Cell 1: LLM generates Cypher using prompt_1 + schema + user question
import os
from neo4j import GraphDatabase
from openai import OpenAI

PROMPT_1_PATH = '/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/agentic_cyph_retr_cycle_sandbox/prompt_1_q1.txt'
# QUESTION = 'Which documents do I need to bring to passport control in France?'
QUESTION = 'Which countries are mentioned in database?'



NEO4J_URI = 'neo4j://localhost:7688'
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = '12345678'

DEEPSEEK_API_KEY = 'sk-8da8031877dc4c39ae8d7b624c70f01f'
DEEPSEEK_MODEL = os.getenv('DEEPSEEK_MODEL', 'deepseek-reasoner')
DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com/v1')

def fetch_schema_text():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        result = session.run('CALL db.schema.visualization()')
        record = result.single()
    driver.close()

    nodes = record['nodes']
    rels = record['relationships']

    labels = sorted({lbl for n in nodes for lbl in n.labels})
    lines = ['NODE LABELS:']
    lines += [f'- {l}' for l in labels]
    lines.append('')
    lines.append('RELATIONSHIPS:')
    for r in rels:
        s = ':'.join(r.start_node.labels) or ''
        e = ':'.join(r.end_node.labels) or ''
        lines.append(f'- (:{s})-[:{r.type}]->(:{e})')
    return ''.join(lines)

def build_prompt(prompt_template, schema_text, question):
    prompt = prompt_template.replace('{SCHEMA_TEXT}', schema_text).replace('{SCHEMA}', schema_text)
    prompt = prompt.replace('{USER_QUERY}', question).replace('{USER_QUESTION}', question)
    if '{SCHEMA_TEXT}' not in prompt_template and '{SCHEMA}' not in prompt_template:
        prompt += f"DATABASE SCHEMA{schema_text}"
    if '{USER_QUERY}' not in prompt_template and '{USER_QUESTION}' not in prompt_template:
        prompt += f"QUESTION {question}"
    return prompt

def call_llm(prompt):
    client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)
    resp = client.chat.completions.create(
        model=DEEPSEEK_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return resp.choices[0].message.content.strip()

with open(PROMPT_1_PATH, 'r', encoding='utf-8') as f:
    prompt_1_template = f.read()

schema_text = fetch_schema_text()
prompt_1 = build_prompt(prompt_1_template, schema_text, QUESTION)
cypher = call_llm(prompt_1)

import json as _json
import re as _re

def extract_cypher(text):
    t = text.strip()
    if t.startswith("```"):
        t = _re.sub(r"^```[a-zA-Z]*\n|\n```$", "", t).strip()
    if t.startswith("{") and "cypher" in t:
        try:
            obj = _json.loads(t)
            if isinstance(obj, dict) and "cypher" in obj:
                return obj["cypher"].strip()
        except Exception:
            pass
    m = _re.search(r'"cypher"\s*:\s*"([^"]+)"', t, _re.S)
    if m:
        return m.group(1).replace("\n", "")
    return t

cypher = extract_cypher(cypher)

print(cypher)
print(prompt_1)
# cypher = '{"cypher":"MATCH (c:Countries) RETURN DISTINCT c LIMIT 200"}'
print(cypher)
import subprocess
subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


MATCH (c:Country) RETURN DISTINCT c LIMIT 200
You are a Neo4j Cypher generator for GraphRAG retrieval.

You will receive:
- Database schema text (labels, relationship types, sample properties).
- A user question.

GOAL
Generate ONE read-only Cypher query that retrieves the facts needed to answer the question.

GUIDELINES
1) Use ONLY labels, relationship types, and properties that appear in the schema text. Do NOT invent anything.
2) Read-only only: MATCH / OPTIONAL MATCH / WHERE / WITH / RETURN / ORDER BY / LIMIT.
   Forbidden: CREATE, MERGE, SET, DELETE, CALL, APOC, LOAD CSV.
3) Prefer returning relationship facts (s)-[p]->(o) when possible.
4) If the question is scoped to a location (city/country/airport), include the relevant location paths if they exist in the schema:
   atCountry OR atCity->locatedInCountry OR atAirport->locatedInCity->locatedInCountry.
5) Add LIMIT 200 (or smaller) if the result could be large.

OUTPUT FORMAT (STRICT)
Return ONLY valid JSON:
{"cypher":"<ONE Cyphe

CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

In [62]:
# ТЕСТИРОВАНИЕ: ошибка c:Country и ДРУГАЯ БУКВА p
cypher = 'MATCH (c:Country) RETURN p LIMIT 200'

In [63]:
# Cell 2: Execute Cypher (split into independent statements)
from neo4j import GraphDatabase

URI = "neo4j://localhost:7689"
USER = "neo4j"
PASSWORD = "12345678"
DATABASE = "neo4j"

def split_statements(text):
    stmts = []
    buf = []
    in_single = False
    in_double = False
    for ch in text:
        if ch == "'" and not in_double:
            in_single = not in_single
        elif ch == '"' and not in_single:
            in_double = not in_double
        if ch == ';' and not in_single and not in_double:
            stmt = ''.join(buf).strip()
            if stmt:
                stmts.append(stmt)
            buf = []
        else:
            buf.append(ch)
    last = ''.join(buf).strip()
    if last:
        stmts.append(last)
    return stmts

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

cypher_statements = split_statements(cypher)
EXEC_ERROR = False
EXEC_ERROR_TEXT = ''
data = None

try:
    with driver.session(database=DATABASE) as session:
        total = len(cypher_statements)
        for i, stmt in enumerate(cypher_statements, 1):
            if i == total:
                data = session.run(stmt).data()
            else:
                session.run(stmt).consume()
    print(f"✅ Executed {len(cypher_statements)} statements.")
    if data is not None:
        print(data)
except Exception as e:
    EXEC_ERROR = True
    EXEC_ERROR_TEXT = str(e)
    print(EXEC_ERROR_TEXT)
finally:
    driver.close()
if EXEC_ERROR:
    print('🟥 ERROR')

{code: Neo.ClientError.Statement.SyntaxError} {message: Variable `p` not defined (line 1, column 26 (offset: 25))
"MATCH (c:Country) RETURN p LIMIT 200"
                          ^}
🟥 ERROR


In [58]:
# Cell 3: If execution error, call LLM fixer and overwrite cypher
from openai import OpenAI

if EXEC_ERROR:
    with open('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/agentic_cyph_retr_cycle_sandbox/prompt_2_reviewer.txt', 'r', encoding='utf-8') as f:
        prompt_2_template = f.read()

    prompt_2 = prompt_2_template.replace('{QUESTION}', QUESTION)
    prompt_2 = prompt_2.replace('{SCHEMA}', schema_text)
    prompt_2 = prompt_2.replace('{CYPHER}', cypher)
    prompt_2 = prompt_2.replace('{ERROR}', 'Execution error')

    client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)
    resp = client.chat.completions.create(
        model=DEEPSEEK_MODEL,
        messages=[{'role': 'user', 'content': prompt_2}],
        temperature=0
    )
    cypher = resp.choices[0].message.content.strip()
    print('ОШИБКА')
    print(cypher)


ОШИБКА
MATCH (c:Country) RETURN c LIMIT 200
